In [9]:
!pip install geopy


   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd

df = pd.read_excel("datasets_final_24h_clean.xlsx")

df.head()

,client_id,client_type,client_name,owner_name,specialty_type,address,governorate,delegation,latitude,longitude,...,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23,day_mean,day_max
0,CLI0158,Cabinet Médical,Pr. Mohamed Elloumi,Pr. Mohamed Elloumi,Pédiatrie,"Centre Médical, 131 Rue de la République, Etta...",Ariana,Ettadhamen,36.849569,10.159108,...,52.47,52.36,0.0,0.0,0.0,0.0,0.0,0.0,13.66,61.97
1,CLI0158,Cabinet Médical,Pr. Mohamed Elloumi,Pr. Mohamed Elloumi,Pédiatrie,"Centre Médical, 131 Rue de la République, Etta...",Ariana,Ettadhamen,36.849569,10.159108,...,42.37,48.03,0.0,0.0,0.0,0.0,0.0,0.0,12.74,59.02
2,CLI0158,Cabinet Médical,Pr. Mohamed Elloumi,Pr. Mohamed Elloumi,Pédiatrie,"Centre Médical, 131 Rue de la République, Etta...",Ariana,Ettadhamen,36.849569,10.159108,...,53.81,46.31,0.0,0.0,0.0,0.0,0.0,0.0,12.24,57.53
3,CLI0158,Cabinet Médical,Pr. Mohamed Elloumi,Pr. Mohamed Elloumi,Pédiatrie,"Centre Médical, 131 Rue de la République, Etta...",Ariana,Ettadhamen,36.849569,10.159108,...,53.01,44.13,0.0,0.0,0.0,0.0,0.0,0.0,13.90,62.70
4,CLI0158,Cabinet Médical,Pr. Mohamed Elloumi,Pr. Mohamed Elloumi,Pédiatrie,"Centre Médical, 131 Rue de la République, Etta...",Ariana,Ettadhamen,36.849569,10.159108,...,51.47,58.36,0.0,0.0,0.0,0.0,0.0,0.0,14.96,73.83


In [3]:
df.columns

Index(['client_id', 'client_type', 'client_name', 'owner_name',
       'specialty_type', 'address', 'governorate', 'delegation', 'latitude',
       'longitude', 'phone', 'patients_per_month', 'last_visit_date',
       'total_visits', 'engagement_level', 'preferred_products', 'languages',
       'day_text', 'is_night_pharmacy', 'is_cabinet', 'hour_0', 'hour_1',
       'hour_2', 'hour_3', 'hour_4', 'hour_5', 'hour_6', 'hour_7', 'hour_8',
       'hour_9', 'hour_10', 'hour_11', 'hour_12', 'hour_13', 'hour_14',
       'hour_15', 'hour_16', 'hour_17', 'hour_18', 'hour_19', 'hour_20',
       'hour_21', 'hour_22', 'hour_23', 'day_mean', 'day_max'],
      dtype='object')

In [11]:
from geopy.distance import geodesic

def compute_distance(row, user_location):
    return geodesic(
        (row['latitude'], row['longitude']),
        user_location
    ).km

In [13]:
def compute_activity(row, start_hour, end_hour):
    hours = [f'hour_{h}' for h in range(start_hour, end_hour)]
    return row[hours].mean()

In [15]:
def prepare_data(df, user_location, start_hour, end_hour):
    
    df = df.copy()
    
    # distance
    df['distance'] = df.apply(lambda row: compute_distance(row, user_location), axis=1)
    
    # activité horaire
    df['activity'] = df.apply(lambda row: compute_activity(row, start_hour, end_hour), axis=1)
    
    return df

In [40]:
def normalize(df):
    
    df['distance_norm'] = 1 / (df['distance'] + 0.001)
    
    df['patients_per_month'] = pd.to_numeric(df['patients_per_month'], errors='coerce')
    df['patients_norm'] = df['patients_per_month'] / df['patients_per_month'].max()
    
    df['engagement_norm'] = df['engagement_score'] / df['engagement_score'].max()
    
    df['activity_norm'] = df['activity'] / df['activity'].max()
    
    return df

In [42]:
def compute_score(df):
    
    w1, w2, w3, w4 = 0.4, 0.2, 0.2, 0.2
    
    df['score'] = (
        w1 * df['distance_norm'] +
        w2 * df['patients_norm'] +
        w3 * df['engagement_norm'] +
        w4 * df['activity_norm']
    )
    
    return df

In [44]:
def filter_data(df, want_pharmacy=True, want_clinic=True):
    
    if want_pharmacy and want_clinic:
        return df
    
    elif want_pharmacy:
        return df[df['is_night_pharmacy'] == 1]
    
    elif want_clinic:
        return df[df['is_cabinet'] == 1]

In [46]:
def select_best(df, n):
    return df.sort_values(by='score', ascending=False).head(n)

In [48]:
df['engagement_level'] = df['engagement_level'].str.strip()

mapping = {
    'Faible': 1,
    'Moyen': 2,
    'Élevé': 3,
    'Eleve': 3  # sécurité
}

df['engagement_score'] = df['engagement_level'].map(mapping)

In [56]:
def compute_traffic_score(activity):
    
    if activity == 0:
        return 0  # fermé
    
    return 1 / (activity + 1)  # stable

In [52]:
user_location = (36.85, 10.20)

# 1. préparation
df_prep = prepare_data(df, user_location, start_hour=8, end_hour=16)

# 2. normalisation
df_norm = normalize(df_prep)

# 3. calcul score
df_scored = compute_score(df_norm)

#  4. SUPPRIMER DOUBLONS (ICI EXACTEMENT)
df_scored = df_scored.sort_values(by='score', ascending=False)
df_unique = df_scored.drop_duplicates(subset='client_id')

# 5. filtrage
df_filtered = filter_data(df_unique, want_pharmacy=True, want_clinic=True)

# 6. sélection finale
best_points = select_best(df_filtered, 5)

# 7. affichage
best_points[['client_name', 'score', 'distance']]

,client_name,score,distance
1544,Para-Pharmacie de l'Étoile,0.727144,0.911530
46,Para-Pharmacie de la Santé,0.704795,0.835273
1589,Pharmacie du Quartier,0.618844,1.400652
74,Para-Pharmacie Moderne,0.561933,2.696382
305,Para-Pharmacie de la Liberté,0.561034,3.802579


In [37]:
df['engagement_level'].unique()

array(['Moyen', 'Faible', 'Élevé'], dtype=object)

In [3]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic

In [5]:
df = pd.read_excel("datasets_final_24h_clean.xlsx")

# supprimer doublons initiaux
df = df.drop_duplicates(subset='client_id')

# convertir patients
df['patients_per_month'] = pd.to_numeric(df['patients_per_month'], errors='coerce')

In [8]:
mapping = {
    'Faible': 1,
    'Moyen': 2,
    'Élevé': 3,
    'Eleve': 3
}

df['engagement_score'] = df['engagement_level'].map(mapping)

In [10]:
def compute_distance(p1, p2):
    return geodesic(p1, p2).km

In [12]:
def get_activity(row, hour):
    hour = int(hour)
    col = f'hour_{hour}'
    
    if col in row:
        return row[col]
    return 0

In [14]:
def compute_traffic_score(activity):
    
    if activity == 0:
        return 0  # fermé
    
    return 1 / (activity + 1)

In [16]:
def compute_dynamic_score(df, user_location, start_hour):
    
    df = df.copy()
    scores = []
    
    for idx, row in df.iterrows():
        
        #  distance
        dist = compute_distance(
            user_location,
            (row['latitude'], row['longitude'])
        )
        distance_score = 1 / (dist + 0.001)
        
        #  activité à l'heure de départ
        activity = get_activity(row, start_hour)
        traffic_score = compute_traffic_score(activity)
        
        #  patients
        patients = row['patients_per_month'] if not pd.isna(row['patients_per_month']) else 0
        patients_norm = patients / df['patients_per_month'].max()
        
        #  engagement
        engagement = row['engagement_score'] if not pd.isna(row['engagement_score']) else 0
        engagement_norm = engagement / df['engagement_score'].max()
        
        #  SCORE FINAL
        score = (
            0.4 * distance_score +
            0.3 * traffic_score +
            0.2 * patients_norm +
            0.1 * engagement_norm
        )
        
        scores.append(score)
    
    df['score'] = scores
    
    return df

In [18]:
def remove_duplicates(df):
    df = df.sort_values(by='score', ascending=False)
    return df.drop_duplicates(subset='client_id')

In [20]:
def select_best(df, n):
    return df.sort_values(by='score', ascending=False).head(n)

In [22]:
def compute_route(df, user_location):
    
    points = df.copy()
    route = []
    current_location = user_location
    
    while len(points) > 0:
        
        points['dist_temp'] = points.apply(
            lambda row: compute_distance(
                current_location,
                (row['latitude'], row['longitude'])
            ),
            axis=1
        )
        
        nearest = points.loc[points['dist_temp'].idxmin()]
        
        route.append(nearest)
        
        current_location = (nearest['latitude'], nearest['longitude'])
        
        points = points.drop(nearest.name)
    
    return pd.DataFrame(route)

In [24]:
#  INPUT utilisateur
user_location = (36.85, 10.20)
start_hour = 9
n_points = 5

#  scoring dynamique
df_scored = compute_dynamic_score(df, user_location, start_hour)

#  supprimer doublons
df_unique = remove_duplicates(df_scored)

#  sélectionner meilleurs
best_points = select_best(df_unique, n_points)

#  calcul trajet
route = compute_route(best_points, user_location)

#  résultat
route[['client_name', 'score', 'latitude', 'longitude']]

,client_name,score,latitude,longitude
42,Para-Pharmacie de la Santé,0.569242,36.848134,10.209073
1589,Pharmacie du Quartier,0.539293,36.856875,10.213171
217,Pr. Leila Amari,0.417548,36.858843,10.211617
1540,Para-Pharmacie de l'Étoile,0.614922,36.849707,10.189786
672,Para-Pharmacie de la Santé,0.398479,36.831797,10.170860


In [27]:
import folium

def create_full_map(df, route, user_location):
    
    m = folium.Map(location=user_location, zoom_start=12)
    
    #  utilisateur
    folium.Marker(
        location=user_location,
        popup="Start (You)",
        icon=folium.Icon(color='red')
    ).add_to(m)
    
    #  TOUS les points du dataset
    for _, row in df.iterrows():
        
        coord = (row['latitude'], row['longitude'])
        
        folium.CircleMarker(
            location=coord,
            radius=3,
            color='gray',
            fill=True,
            fill_opacity=0.5,
            popup=row['client_name']
        ).add_to(m)
    
    #  TRAJET OPTIMAL
    route_coords = [user_location]
    
    for i, row in route.iterrows():
        
        coord = (row['latitude'], row['longitude'])
        route_coords.append(coord)
        
        #  points du trajet en bleu/vert
        color = 'green' if row['score'] > 0.7 else 'blue'
        
        popup = f"""
        <b>{i+1} - {row['client_name']}</b><br>
        Score: {round(row['score'], 2)}
        """
        
        folium.Marker(
            location=coord,
            popup=popup,
            icon=folium.Icon(color=color)
        ).add_to(m)
    
    #  ligne du trajet
    folium.PolyLine(route_coords, color="green", weight=4).add_to(m)
    
    return m

In [29]:
m = create_full_map(df, route, user_location)
m

In [32]:
m.save("map_route.html")

In [1]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import folium
from folium.plugins import MarkerCluster

In [3]:
df = pd.read_excel("datasets_final_24h_clean.xlsx")

# supprimer doublons
df = df.drop_duplicates(subset='client_id')

# supprimer NaN coords
df = df.dropna(subset=['latitude', 'longitude'])

# convertir coords
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# filtrer Tunisie
df = df[
    (df['latitude'] >= 30) & (df['latitude'] <= 38) &
    (df['longitude'] >= 7) & (df['longitude'] <= 12)
]

# enlever mer (approx Tunis)
df = df[
    (df['latitude'] > 36.7) &
    (df['longitude'] > 10.0)
]

# convertir patients
df['patients_per_month'] = pd.to_numeric(df['patients_per_month'], errors='coerce')

In [5]:
mapping = {
    'Faible': 1,
    'Moyen': 2,
    'Élevé': 3,
    'Eleve': 3
}

df['engagement_score'] = df['engagement_level'].map(mapping)

In [7]:
def compute_distance(p1, p2):
    return geodesic(p1, p2).km

In [9]:
def get_activity(row, hour):
    col = f'hour_{int(hour)}'
    return row[col] if col in row else 0
def compute_traffic_score(activity):
    if activity == 0:
        return 0
    return 1 / (activity + 1)

In [11]:
def compute_dynamic_score(df, user_location, start_hour):
    
    df = df.copy()
    scores = []
    
    for _, row in df.iterrows():
        
        # distance
        dist = compute_distance(user_location, (row['latitude'], row['longitude']))
        distance_score = 1 / (dist + 0.001)
        
        # trafic
        activity = get_activity(row, start_hour)
        traffic_score = compute_traffic_score(activity)
        
        # patients
        patients = row['patients_per_month'] if not pd.isna(row['patients_per_month']) else 0
        patients_norm = patients / df['patients_per_month'].max()
        
        # engagement
        engagement = row['engagement_score'] if not pd.isna(row['engagement_score']) else 0
        engagement_norm = engagement / df['engagement_score'].max()
        
        # score final
        score = (
            0.4 * distance_score +
            0.3 * traffic_score +
            0.2 * patients_norm +
            0.1 * engagement_norm
        )
        
        scores.append(score)
    
    df['score'] = scores
    return df

In [13]:
def remove_duplicates(df):
    df = df.sort_values(by='score', ascending=False)
    return df.drop_duplicates(subset='client_id')

In [15]:
def select_best(df, n):
    return df.sort_values(by='score', ascending=False).head(n)

In [17]:
def compute_route(df, user_location):
    
    points = df.copy()
    route = []
    current_location = user_location
    
    while len(points) > 0:
        
        points['dist_temp'] = points.apply(
            lambda row: compute_distance(current_location, (row['latitude'], row['longitude'])),
            axis=1
        )
        
        nearest = points.loc[points['dist_temp'].idxmin()]
        
        route.append(nearest)
        
        current_location = (nearest['latitude'], nearest['longitude'])
        
        points = points.drop(nearest.name)
    
    return pd.DataFrame(route)

In [19]:
def get_best_route(df, user_location, start_hour, n_points):
    
    df_scored = compute_dynamic_score(df, user_location, start_hour)
    
    df_unique = remove_duplicates(df_scored)
    
    best_points = select_best(df_unique, n_points)
    
    route = compute_route(best_points, user_location)
    
    return route

In [21]:
def create_full_map(df, route, user_location):
    
    m = folium.Map(location=user_location, zoom_start=12)
    
    # utilisateur
    folium.Marker(
        location=user_location,
        popup="Start",
        icon=folium.Icon(color='red')
    ).add_to(m)
    
    # cluster pour tous les points
    cluster = MarkerCluster().add_to(m)
    
    for _, row in df.iterrows():
        folium.CircleMarker(
            location=(row['latitude'], row['longitude']),
            radius=3,
            color='gray',
            fill=True
        ).add_to(cluster)
    
    # trajet
    route_coords = [user_location]
    
    for i, row in route.iterrows():
        
        coord = (row['latitude'], row['longitude'])
        route_coords.append(coord)
        
        color = 'green' if row['score'] > 0.7 else 'blue'
        
        popup = f"""
        <b>{i+1} - {row['client_name']}</b><br>
        Score: {round(row['score'],2)}
        """
        
        folium.Marker(
            location=coord,
            popup=popup,
            icon=folium.Icon(color=color)
        ).add_to(m)
    
    folium.PolyLine(route_coords, color="green", weight=4).add_to(m)
    
    return m

In [23]:
user_location = (36.85, 10.20)
start_hour = 10   # 👈 utilisateur choisit
n_points = 5

route = get_best_route(df, user_location, start_hour, n_points)

m = create_full_map(df, route, user_location)

m

In [25]:
pip install streamlit folium streamlit-folium

   ---------------------------------------- 0.0/529.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/529.2 kB ? eta -:--:--
   ------------------- -------------------- 262.1/529.2 kB ? eta -:--:--
   ------------------- -------------------- 262.1/529.2 kB ? eta -:--:--
   ---------------------------------------- 529.2/529.2 kB 853.3 kB/s  0:00:00

  Attempting uninstall: protobuf

    Found existing installation: protobuf 6.33.0

    Uninstalling protobuf-6.33.0:

      Successfully uninstalled protobuf-6.33.0

   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 2/2 [streamlit-folium]

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
